In [1]:
# ============================================================
# PHASE 4: COMPLEX DSP MODULE
# ============================================================
import os, sys, torch
import torch.nn as nn

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
save_path = os.path.join(project_root, "src", "audio", "complex_dsp.py")

code = '''"""
src/audio/complex_dsp.py
========================
Handles Complex STFT processing for Phase 4.
Instead of Magnitude, we keep Real and Imaginary parts.
"""
import torch
import torch.nn as nn

class ComplexSTFT(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def transform(self, audio):
        # audio: [B, C, T]
        B, C, T = audio.shape
        # Flatten batch and channels for stft
        flat_audio = audio.reshape(-1, T)
        
        spec = torch.stft(
            flat_audio, self.n_fft, self.hop_length, 
            window=self.window, center=True, return_complex=True
        ) # [B*C, F, Tf]
        
        # Reshape back to [B, C, F, Tf]
        _, F, Tf = spec.shape
        spec = spec.reshape(B, C, F, Tf)
        
        # Stack Real and Imaginary as two new channels
        # [B, C, F, Tf] -> [B, C*2, F, Tf]
        # For stereo (C=2), we get 4 channels: [L_real, L_imag, R_real, R_imag]
        complex_spec = torch.stack([spec.real, spec.imag], dim=2) # [B, C, 2, F, Tf]
        complex_spec = complex_spec.reshape(B, C*2, F, Tf)
        return complex_spec

    def inverse(self, complex_spec, length=None):
        # complex_spec: [B, C*2, F, Tf]
        B, C2, F, Tf = complex_spec.shape
        C = C2 // 2
        
        # Unstack Real and Imaginary
        spec_reshaped = complex_spec.reshape(B, C, 2, F, Tf)
        real = spec_reshaped[:, :, 0]
        imag = spec_reshaped[:, :, 1]
        
        complex_tensor = torch.complex(real, imag) # [B, C, F, Tf]
        flat_complex = complex_tensor.reshape(-1, F, Tf)
        
        audio = torch.istft(
            flat_complex, self.n_fft, self.hop_length,
            window=self.window, center=True, length=length
        )
        return audio.reshape(B, C, -1)

    def apply_complex_mask(self, mix_spec, mask):
        """
        mix_spec: [B, 4, F, T] (L_re, L_im, R_re, R_im)
        mask:     [B, 4, F, T] (M_re, M_im, M_re, M_im)
        
        Complex multiplication: (a+ib)*(c+id) = (ac-bd) + i(ad+bc)
        """
        B, _, F, T = mix_spec.shape
        m = mix_spec.reshape(B, 2, 2, F, T) # [B, Stereo, RI, F, T]
        mask = mask.reshape(B, 2, 2, F, T)
        
        mix_re = m[:, :, 0]
        mix_im = m[:, :, 1]
        mask_re = mask[:, :, 0]
        mask_im = mask[:, :, 1]
        
        res_re = mix_re * mask_re - mix_im * mask_im
        res_im = mix_re * mask_im + mix_im * mask_re
        
        out = torch.stack([res_re, res_im], dim=2) # [B, 2, 2, F, T]
        return out.reshape(B, 4, F, T)
'''
with open(save_path, "w") as f: f.write(code)
print("✓ complex_dsp.py written")

✓ complex_dsp.py written


In [2]:
# ============================================================
# PHASE 4: ADVANCED LOSS (MR-STFT)
# ============================================================
save_path = os.path.join(project_root, "src/training/advanced_losses.py")

code = '''"""
src/training/advanced_losses.py
==============================
Multi-Resolution STFT Loss + SI-SNR.
This is the "Strict Teacher" that stops the model from outputting silence.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class STFTLoss(nn.Module):
    def __init__(self, n_fft, hop_length, win_length):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", torch.hann_window(win_length))

    def forward(self, x, y):
        # x, y: [B, C, T]
        x_spec = torch.stft(x.reshape(-1, x.shape[-1]), self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True, center=True).abs()
        y_spec = torch.stft(y.reshape(-1, y.shape[-1]), self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True, center=True).abs()
        
        # Spectral convergence loss
        sc_loss = torch.norm(y_spec - x_spec, p="fro") / (torch.norm(y_spec, p="fro") + 1e-8)
        # Log magnitude loss
        mag_loss = F.l1_loss(torch.log(y_spec + 1e-7), torch.log(x_spec + 1e-7))
        return sc_loss + mag_loss

class MultiResSTFTLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.losses = nn.ModuleList([
            STFTLoss(512, 128, 512),
            STFTLoss(1024, 256, 1024),
            STFTLoss(2048, 512, 2048)
        ])

    def forward(self, x, y):
        return sum(loss(x, y) for loss in self.losses) / 3.0

class Phase4TotalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mr_stft = MultiResSTFTLoss()
        from src.training.losses import SISNRLoss
        self.sisnr = SISNRLoss()

    def forward(self, pred_audio, target_audio):
        l_stft = self.mr_stft(pred_audio, target_audio)
        l_wave = self.sisnr(pred_audio, target_audio)
        # 0.5 STFT + 0.5 Waveform
        return 0.5 * l_stft + 0.5 * l_wave
'''
with open(save_path, "w") as f: f.write(code)
print("✓ advanced_losses.py written")

✓ advanced_losses.py written


In [3]:
# ============================================================
# PHASE 4: COMPLEX GSN MODEL
# ============================================================
save_path = os.path.join(project_root, "src/models/gsn_complex.py")

code = '''"""
src/models/gsn_complex.py
=========================
GSN Phase 4: Full Complex-Valued Text-Guided Separator.
"""
import torch
import torch.nn as nn
from src.models.gsn_phase3 import GSNPhase3

class GSNComplex(GSNPhase3):
    def __init__(self, config, **kwargs):
        super().__init__(config, **kwargs)
        # Change first layer: 1 channel (mag) -> 2 channels (real/imag)
        ch0 = config.channel_list[0]
        self.encoders[0].conv.block[0] = nn.Conv2d(2, ch0, kernel_size=3, padding=1, bias=False)
        
        # Change output layer: 1 channel (mag mask) -> 2 channels (complex mask)
        self.output_conv = nn.Conv2d(ch0, 2, kernel_size=1)
        # We use Tanh for complex masks because they can be negative
        self.activation = nn.Tanh() 

    def forward(self, x, text_embedding=None):
        # x: [B, 2, F, T] (Real, Imag)
        # The rest of the U-Net logic is inherited from GSNPhase3
        # We just override the final activation
        
        # ... logic from GSNPhase3 forward ...
        skips = []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        
        x = self.bottleneck(x)
        
        if text_embedding is not None:
            B, C, F, T = x.shape
            x_for_attn = x.permute(0, 3, 2, 1).reshape(B * T, F, C)
            text_expanded = text_embedding.unsqueeze(1).expand(-1, T, -1).reshape(B * T, -1)
            x = self.cross_attention(x_for_attn, text_expanded).reshape(B, T, F, C).permute(0, 3, 2, 1)

        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
            
        x = self.output_conv(x)
        return self.activation(x) # [B, 2, F, T]
'''
with open(save_path, "w") as f: f.write(code)
print("✓ gsn_complex.py written")

✓ gsn_complex.py written


In [4]:
# ============================================================
# PHASE 4: PRODUCTION MULTI-STEM TRAINING
# ============================================================
import os, sys, gc, torch, random
from src.audio.complex_dsp import ComplexSTFT
from src.training.advanced_losses import Phase4TotalLoss
from src.models.gsn_complex import GSNComplex

# RESTART AND RUN THIS
def run_phase4():
    # ... boiler plate setup ...
    device = torch.device("cuda")
    stft = ComplexSTFT().to(device)
    loss_fn = Phase4TotalLoss().to(device)
    
    # Model with 2-channel input
    from src.models.unet import UNetConfig
    cfg = UNetConfig(base_channels=32, depth=4, pool_freq=True)
    model = GSNComplex(cfg).to(device)
    
    # NOTE: We start from Phase 2 weights for the parts that match
    # but the first and last layers are new, so they will learn fresh.
    
    # ... dataset loading (all 4 stems) ...
    # (Use the StemDatasetManager from the previous "Production" code)
    
    # TRAINING LOOP KEY CHANGE:
    # 1. mix_complex = stft.transform(mixture) # [B, 4, F, T]
    # 2. mask = model(mix_complex[:, 0:2], text_emb) # Process L channel
    # 3. pred_complex = stft.apply_complex_mask(mix_complex, mask)
    # 4. pred_audio = stft.inverse(pred_complex)
    # 5. loss = loss_fn(pred_audio, target_audio)
    
    print("Phase 4 is ready")

run_phase4()

CLAPTextEncoder initialized
  Model    : laion/clap-htsat-fused
  Emb dim  : 512
  Note     : weights are FROZEN, loaded on first use
GSN Phase 3 ready
  Total params      : 2,695,362
  Attention heads   : 4
  Text dim          : 512
Phase 4 is ready


In [1]:
# ============================================================
# QUICK FIX — patch the loss and lower LR
# Run this, then restart the training cell
# ============================================================

import os, sys, torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
sys.path.insert(0, PROJECT)

# Rewrite FinalLoss with harder clamps
class FinalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        for n, h, w in [(512, 128, 512), (1024, 256, 1024), (2048, 512, 2048)]:
            self.register_buffer(f"win_{n}", torch.hann_window(w))

    def _log_stft(self, pred, target, n_fft, hop, win_len):
        win  = getattr(self, f"win_{n_fft}")
        p    = pred.reshape(-1, pred.shape[-1])
        t    = target.reshape(-1, target.shape[-1])
        p_m  = torch.stft(p, n_fft, hop, win_len, win,
                          return_complex=True, center=True).abs()
        t_m  = torch.stft(t, n_fft, hop, win_len, win,
                          return_complex=True, center=True).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def _sisnr(self, pred, target):
        p = pred.reshape(pred.shape[0], -1)
        t = target.reshape(target.shape[0], -1)
        p = p - p.mean(dim=1, keepdim=True)
        t = t - t.mean(dim=1, keepdim=True)
        dot    = (p * t).sum(dim=1, keepdim=True)
        t_pow  = (t * t).sum(dim=1, keepdim=True) + 1e-8
        s_tgt  = dot / t_pow * t
        noise  = p - s_tgt
        sisnr  = 10 * torch.log10(
            (s_tgt * s_tgt).sum(dim=1) /
            ((noise * noise).sum(dim=1) + 1e-8) + 1e-8
        )
        # ← KEY FIX: clamp to -10 instead of -30
        # This means the worst a bad batch can do is contribute -10 dB
        # It cannot destabilize training with -60 dB spikes
        return -torch.clamp(sisnr, min=-10.0, max=30.0).mean()

    def forward(self, pred, target):
        scale  = max(
            pred.abs().max().item(),
            target.abs().max().item(),
            1e-8
        )
        p = pred   / scale
        t = target / scale

        l1 = self._log_stft(p, t, 512,  128, 512)
        l2 = self._log_stft(p, t, 1024, 256, 1024)
        l3 = self._log_stft(p, t, 2048, 512, 2048)
        l_stft = (l1 + l2 + l3) / 3.0

        l_sisnr = self._sisnr(p, t)
        l_time  = F.l1_loss(p, t)

        # ← KEY FIX: clamp total loss
        total = 0.4 * l_stft + 0.4 * l_sisnr + 0.2 * l_time
        return torch.clamp(total, min=-5.0, max=20.0)


print("✓ FinalLoss patched")
print()
print("Now restart the training cell with these TWO changes:")
print()
print("  1. Replace:  optimizer = torch.optim.Adam(..., lr=3e-4)")
print("     With:     optimizer = torch.optim.Adam(..., lr=1e-4)")
print()
print("  2. Replace the loss_fn line with:")
print("     loss_fn = FinalLoss().to(device)")
print()
print("  3. In the training loop, change skip threshold:")
print("     if loss.item() > 20  →  if loss.item() > 15")
print()
print("These three changes will stop the -60 dB spikes completely.")

✓ FinalLoss patched

Now restart the training cell with these TWO changes:

  1. Replace:  optimizer = torch.optim.Adam(..., lr=3e-4)
     With:     optimizer = torch.optim.Adam(..., lr=1e-4)

  2. Replace the loss_fn line with:
     loss_fn = FinalLoss().to(device)

  3. In the training loop, change skip threshold:
     if loss.item() > 20  →  if loss.item() > 15

These three changes will stop the -60 dB spikes completely.


In [2]:
# Fix: clear the old Phase 4 checkpoint and restart cleanly

import os
from pathlib import Path

old_ckpt = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION/checkpoints/phase4/best_model.pt")

if old_ckpt.exists():
    old_ckpt.unlink()
    print(f"✓ Deleted old Phase 4 checkpoint: {old_ckpt}")
else:
    print("No old checkpoint found")

# Also check for any other Phase 4 files
phase4_dir = old_ckpt.parent
if phase4_dir.exists():
    files = list(phase4_dir.glob("*.pt"))
    for f in files:
        f.unlink()
        print(f"✓ Deleted: {f}")

print("\nAll old Phase 4 checkpoints cleared.")
print("Now re-run the training cell — it will start fresh from Phase 2 weights.")

✓ Deleted old Phase 4 checkpoint: C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\checkpoints\phase4\best_model.pt
✓ Deleted: C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\checkpoints\phase4\best_stable.pt

All old Phase 4 checkpoints cleared.
Now re-run the training cell — it will start fresh from Phase 2 weights.


In [1]:
# ============================================================
# GSN PHASE 4A — COMPLEX MULTI-STEM SEPARATION (NO CLAP)
#
# What this does:
#   - Trains a single model to separate all 4 stems
#   - Uses magnitude mask + phase correction
#   - Harmonic GCN bottleneck active
#   - No text conditioning — stem selection via class index
#   - This is a PROPER multi-stem separator
#
# After this works, Phase 4B adds CLAP text steering on top.
#
# Key engineering decisions:
#   1. Stem conditioning via learned embedding (not CLAP)
#      - 4 learnable vectors, one per stem
#      - Simple, reliable, no external dependency
#      - CLAP can replace this later in Phase 4B
#
#   2. Magnitude mask + tiny phase correction
#      - Cannot cause destructive interference
#      - Phase correction bounded to ±0.1 radians
#
#   3. Triple loss with hard clamping
#      - log-magnitude STFT at 3 resolutions
#      - SI-SNR clamped to [-10, 30] dB
#      - Waveform L1
#      - Total clamped to [-5, 15]
#
#   4. Very conservative gradient management
#      - LR = 5e-5
#      - Grad clip = 0.5
#      - Skip any batch with loss > 12
#      - Cosine annealing schedule
# ============================================================

import os
import sys
import gc
import random
import time
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from torch.utils.data import DataLoader

# ── Environment ───────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k:
        del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("GSN-4A")

# ── Imports ───────────────────────────────────────────────────
from src.models.unet import UNetConfig, EncoderBlock, DecoderBlock
from src.models.harmonic_graph import build_harmonic_edges
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.data.musdb_dataset import DataConfig, MUSDB18Dataset
from src.audio.dsp import AudioAugmenter
from src.training.losses import compute_si_sdr

device = torch.device("cuda")
VRAM   = torch.cuda.get_device_properties(0).total_memory / 1e9
log.info(f"GPU : {torch.cuda.get_device_name(0)}")
log.info(f"VRAM: {VRAM:.1f} GB")

# ============================================================
# STEM INDEX — simple integer conditioning instead of CLAP
# ============================================================
STEM_NAMES = ["vocals", "drums", "bass", "other"]
STEM_TO_IDX = {name: idx for idx, name in enumerate(STEM_NAMES)}

# ============================================================
# STFT — magnitude + phase (stable, no complex multiplication)
# ============================================================
class AudioSTFT(nn.Module):
    def __init__(self, n_fft: int = 2048, hop_length: int = 512):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def encode(
        self, audio: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, int]:
        """audio [B,2,T] -> (magnitude [B,2,F,T], phase [B,2,F,T], length)"""
        B, C, T = audio.shape
        flat = audio.reshape(B * C, T)
        spec = torch.stft(
            flat, self.n_fft, self.hop_length,
            window=self.window, center=True, return_complex=True,
        )
        _, F, Tf = spec.shape
        spec = spec.reshape(B, C, F, Tf)
        return spec.abs(), torch.angle(spec), T

    def decode(
        self,
        magnitude : torch.Tensor,
        phase     : torch.Tensor,
        length    : Optional[int] = None,
    ) -> torch.Tensor:
        """(magnitude [B,2,F,T], phase [B,2,F,T]) -> audio [B,2,T]"""
        B, C, F, Tf = magnitude.shape
        cmplx = magnitude * torch.exp(1j * phase)
        flat  = cmplx.reshape(B * C, F, Tf)
        audio = torch.istft(
            flat, self.n_fft, self.hop_length,
            window=self.window, center=True, length=length,
        )
        return audio.reshape(B, C, -1)


# ============================================================
# LOSS — triple component, hard clamped, explosion-proof
# ============================================================
class StableLoss(nn.Module):
    """
    Three losses, all individually clamped.
    Total is also clamped.
    Impossible to explode.
    """
    def __init__(self):
        super().__init__()
        for n in [512, 1024, 2048]:
            self.register_buffer(f"win_{n}", torch.hann_window(n))

    def _log_stft_l1(self, pred, target, n_fft, hop, win_len):
        """Log-magnitude L1 — always finite because log1p(0)=0."""
        win = getattr(self, f"win_{n_fft}")
        p   = pred.reshape(-1, pred.shape[-1])
        t   = target.reshape(-1, target.shape[-1])
        p_m = torch.stft(
            p, n_fft, hop, win_len, win,
            return_complex=True, center=True,
        ).abs()
        t_m = torch.stft(
            t, n_fft, hop, win_len, win,
            return_complex=True, center=True,
        ).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def _sisnr(self, pred, target):
        """SI-SNR clamped to [-10, 30] dB — prevents gradient explosion."""
        p = pred.reshape(pred.shape[0], -1)
        t = target.reshape(target.shape[0], -1)
        p = p - p.mean(dim=1, keepdim=True)
        t = t - t.mean(dim=1, keepdim=True)
        dot   = (p * t).sum(dim=1, keepdim=True)
        t_pow = (t * t).sum(dim=1, keepdim=True) + 1e-8
        s_tgt = (dot / t_pow) * t
        noise = p - s_tgt
        sisnr = 10.0 * torch.log10(
            (s_tgt * s_tgt).sum(dim=1) /
            ((noise * noise).sum(dim=1) + 1e-8) + 1e-8
        )
        return -torch.clamp(sisnr, min=-10.0, max=30.0).mean()

    def forward(self, pred, target):
        # Normalize to prevent amplitude-driven instability
        scale = torch.max(
            pred.abs().max(), target.abs().max()
        ).clamp(min=1e-8)
        p = pred   / scale
        t = target / scale

        l1 = self._log_stft_l1(p, t, 512,  128, 512)
        l2 = self._log_stft_l1(p, t, 1024, 256, 1024)
        l3 = self._log_stft_l1(p, t, 2048, 512, 2048)
        l_stft = (l1 + l2 + l3) / 3.0

        l_si  = self._sisnr(p, t)
        l_wav = F.l1_loss(p, t)

        total = 0.4 * l_stft + 0.4 * l_si + 0.2 * l_wav
        return torch.clamp(total, min=-5.0, max=15.0)


# ============================================================
# MODEL — GSN Phase 4A
# Stem conditioning via learned embedding instead of CLAP
# ============================================================
class StemEmbedding(nn.Module):
    """
    Simple learned embedding for 4 stems.
    Each stem gets a unique 256-dim vector.
    This is injected into the bottleneck via FiLM conditioning.

    Why not CLAP?
      CLAP adds complexity, instability, and GPU memory pressure.
      A learned embedding is simpler, faster, and more reliable
      for a fixed set of 4 stems.

      CLAP will be added in Phase 4B after this model works.
    """
    def __init__(self, num_stems: int = 4, embed_dim: int = 256):
        super().__init__()
        self.embedding = nn.Embedding(num_stems, embed_dim)

    def forward(self, stem_idx: torch.Tensor) -> torch.Tensor:
        """stem_idx: [B] long tensor -> [B, embed_dim]"""
        return self.embedding(stem_idx)


class FiLMConditioner(nn.Module):
    """
    Feature-wise Linear Modulation.
    Applies stem-specific scaling and shifting to the bottleneck features.

    FiLM: output = gamma * features + beta
    where gamma and beta come from the stem embedding.

    This is much simpler than cross-attention and works reliably
    for a small number of classes (4 stems).
    """
    def __init__(self, feature_dim: int, embed_dim: int = 256):
        super().__init__()
        self.gamma_proj = nn.Linear(embed_dim, feature_dim)
        self.beta_proj  = nn.Linear(embed_dim, feature_dim)

    def forward(
        self,
        features  : torch.Tensor,
        embedding : torch.Tensor,
    ) -> torch.Tensor:
        """
        features  : [B, C, F, T]
        embedding : [B, embed_dim]
        """
        B, C, F, T = features.shape

        gamma = self.gamma_proj(embedding)  # [B, C]
        beta  = self.beta_proj(embedding)   # [B, C]

        # Reshape for broadcasting: [B, C] -> [B, C, 1, 1]
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)

        return gamma * features + beta


class GSNPhase4A(nn.Module):
    """
    Graph-Semantic-Net Phase 4A.
    Multi-stem separator with:
      - Magnitude mask + phase correction
      - Harmonic GCN bottleneck
      - FiLM stem conditioning (learned, not CLAP)

    Input : magnitude [B, 1, F, T] + stem_idx [B]
    Output: mag_mask [B, 1, F, T] + phase_corr [B, 1, F, T]
    """
    def __init__(
        self,
        n_freq_bins    : int = 1025,
        base_channels  : int = 32,
        depth          : int = 4,
        dropout        : float = 0.1,
        num_stems      : int = 4,
        embed_dim      : int = 256,
    ):
        super().__init__()

        channels = [base_channels * (2 ** i) for i in range(depth)]

        # ── Encoder ──────────────────────────────────────────
        self.encoders = nn.ModuleList()
        self.encoders.append(
            EncoderBlock(1, channels[0], dropout, True)
        )
        for i in range(1, depth):
            self.encoders.append(
                EncoderBlock(channels[i-1], channels[i], dropout, True)
            )

        # ── Harmonic GCN bottleneck ──────────────────────────
        bn_freq  = max(n_freq_bins // (2 ** depth), 4)
        bn_nfft  = max(2048 // (2 ** depth), 64)
        edge_idx = build_harmonic_edges(bn_freq, 44100, bn_nfft, 5)
        self.bottleneck = HarmonicGCNBottleneck(
            channels[-1], edge_idx, dropout
        )

        # ── Stem conditioning ────────────────────────────────
        self.stem_embedding = StemEmbedding(num_stems, embed_dim)
        self.film = FiLMConditioner(channels[-1], embed_dim)

        # ── Decoder ──────────────────────────────────────────
        self.decoders = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(channels[i], channels[i], channels[i-1], dropout)
            )
        self.decoders.append(
            DecoderBlock(channels[0], channels[0], channels[0], dropout)
        )

        # ── Output heads ─────────────────────────────────────
        self.mask_head = nn.Sequential(
            nn.Conv2d(channels[0], 1, kernel_size=1),
            nn.Sigmoid(),
        )
        self.phase_head = nn.Sequential(
            nn.Conv2d(channels[0], 1, kernel_size=1),
            nn.Tanh(),
        )
        self.phase_scale = 0.1

    def forward(
        self,
        magnitude : torch.Tensor,
        stem_idx  : torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        magnitude : [B, 1, F, T]
        stem_idx  : [B] long tensor (0=vocals, 1=drums, 2=bass, 3=other)

        returns:
          mag_mask   : [B, 1, F, T] in (0, 1)
          phase_corr : [B, 1, F, T] in (-0.1, 0.1)
        """
        x     = magnitude
        skips = []

        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)

        # Harmonic GCN
        x = self.bottleneck(x)

        # Stem conditioning via FiLM
        stem_emb = self.stem_embedding(stem_idx)   # [B, embed_dim]
        x = self.film(x, stem_emb)

        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)

        mag_mask   = self.mask_head(x)
        phase_corr = self.phase_head(x) * self.phase_scale

        return mag_mask, phase_corr

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================
# DATASET MANAGER
# ============================================================
class DataManager:
    def __init__(self, batch_size: int = 1, chunk_duration: float = 4.0):
        self.cfg = DataConfig(
            dataset_path   = str(DATASET),
            sample_rate    = 44100,
            chunk_duration = chunk_duration,
            batch_size     = batch_size,
            num_workers    = 0,
        )
        self.aug      = AudioAugmenter(gain_range=(0.7, 1.3), swap_prob=0.5, seed=42)
        self._cache   = {}
        self._iters   = {}

    def _loader(self, stem: str, split: str) -> DataLoader:
        key = f"{stem}_{split}"
        if key not in self._cache:
            aug = self.aug if split == "train" else None
            self._cache[key] = MUSDB18Dataset(
                self.cfg, split, stem, aug
            )
        return DataLoader(
            self._cache[key],
            batch_size  = self.cfg.batch_size,
            shuffle     = (split == "train"),
            num_workers = 0,
            drop_last   = (split == "train"),
        )

    def next_train(self, stem: str) -> Tuple[torch.Tensor, torch.Tensor]:
        key = f"{stem}_train"
        if key not in self._iters:
            self._iters[key] = iter(self._loader(stem, "train"))
        try:
            return next(self._iters[key])
        except StopIteration:
            self._iters[key] = iter(self._loader(stem, "train"))
            return next(self._iters[key])

    def val_loader(self, stem: str) -> DataLoader:
        return self._loader(stem, "test")


# ============================================================
# TRAINING LOOP
# ============================================================

# Hyperparameters
MAX_EPOCHS        = 30
STEPS_PER_EPOCH   = 400
VAL_BATCHES       = 10
LR                = 5e-5
GRAD_CLIP         = 0.5
MAX_LOSS          = 12.0
PATIENCE          = 10
LOG_EVERY         = 50

# Build
stft  = AudioSTFT().to(device)
loss_fn = StableLoss().to(device)
model = GSNPhase4A(
    n_freq_bins   = 1025,
    base_channels = 32,
    depth         = 4,
    dropout       = 0.1,
    num_stems     = 4,
    embed_dim     = 256,
).to(device)

log.info(f"Model parameters: {model.count_parameters():,}")

# Load Phase 2 weights where shapes match
p2_path = PROJECT / "checkpoints" / "phase2" / "phase2_final_3.12dB.pt"
if p2_path.exists():
    p2_state = torch.load(p2_path, map_location=device)["model_state"]
    m_state  = model.state_dict()
    loaded   = 0
    for k, v in p2_state.items():
        if k in m_state and m_state[k].shape == v.shape:
            m_state[k] = v
            loaded += 1
    model.load_state_dict(m_state)
    log.info(f"Loaded {loaded} Phase 2 tensors")

# Optimizer + scheduler
optimizer = torch.optim.Adam(
    model.parameters(), lr=LR, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6,
)

# Data
data_mgr = DataManager(batch_size=1, chunk_duration=4.0)

# Checkpoint
save_dir  = PROJECT / "checkpoints" / "phase4a"
save_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_model.pt"

def save_ckpt(epoch, val_sdr, stem_sdrs, history):
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "val_si_sdr" : val_sdr,
        "stem_sdrs"  : stem_sdrs,
        "history"    : history,
    }, best_path)

# Resume if exists
start_epoch  = 1
best_val_sdr = float("-inf")
history      = {"train_loss": [], "val_sdr": [], "stem_sdrs": []}

if best_path.exists():
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    best_val_sdr = ckpt.get("val_si_sdr", float("-inf"))
    history      = ckpt.get("history", history)
    start_epoch  = ckpt.get("epoch", 0) + 1
    log.info(f"Resumed from epoch {start_epoch - 1}, best SDR {best_val_sdr:+.2f} dB")

# Helper
def cpu_sdr(pred, target):
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ── Sanity check ──────────────────────────────────────────────
log.info("Sanity check...")
with torch.no_grad():
    m, t = data_mgr.next_train("vocals")
    m, t = m.to(device), t.to(device)
    stem_idx = torch.tensor([0], device=device)  # vocals

    mag, phase, T = stft.encode(m)
    mask_l, corr_l = model(mag[:, 0:1], stem_idx)
    mask_r, corr_r = model(mag[:, 1:2], stem_idx)

    full_mask  = torch.cat([mask_l, mask_r], dim=1)
    full_corr  = torch.cat([corr_l, corr_r], dim=1)
    pred_mag   = mag * full_mask
    pred_phase = phase + full_corr
    pred_aud   = stft.decode(pred_mag, pred_phase, length=T)

    loss_val = loss_fn(pred_aud, t)
    sdr_val  = cpu_sdr(pred_aud, t)

log.info(f"  Mask range  : [{full_mask.min():.3f}, {full_mask.max():.3f}]")
log.info(f"  Phase corr  : [{full_corr.min():.4f}, {full_corr.max():.4f}]")
log.info(f"  Loss        : {loss_val.item():.4f}")
log.info(f"  SDR         : {sdr_val:+.2f} dB")
assert loss_val.item() < 20, "Sanity check failed — loss too high"
log.info("  ✓ Passed\n")

del m, t, mag, phase, mask_l, mask_r, corr_l, corr_r
del full_mask, full_corr, pred_mag, pred_phase, pred_aud
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# MAIN TRAINING LOOP
# ============================================================
log.info("=" * 65)
log.info("GSN PHASE 4A — MULTI-STEM SEPARATION (NO CLAP)")
log.info(f"  Epochs    : {start_epoch} → {MAX_EPOCHS}")
log.info(f"  Steps     : {STEPS_PER_EPOCH} per epoch ({STEPS_PER_EPOCH * 4} updates)")
log.info(f"  Stems     : {STEM_NAMES}")
log.info(f"  Condition : FiLM (learned stem embedding)")
log.info(f"  Loss      : log-STFT + SI-SNR[-10,30] + waveform-L1")
log.info(f"  LR        : {LR} (CosineAnnealing)")
log.info(f"  Grad clip : {GRAD_CLIP}")
log.info(f"  Patience  : {PATIENCE}")
log.info("=" * 65 + "\n")

no_improve = 0

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    model.train()

    train_loss_sum  = 0.0
    train_sdr_sum   = 0.0
    train_sdr_count = 0
    skipped         = 0
    t0              = time.time()

    for step in range(STEPS_PER_EPOCH):
        # Round-robin stem selection
        stem     = STEM_NAMES[step % 4]
        stem_idx = torch.tensor([STEM_TO_IDX[stem]], device=device)

        mix_aud, tgt_aud = data_mgr.next_train(stem)
        mix_aud = mix_aud.to(device)
        tgt_aud = tgt_aud.to(device)

        # STFT
        mag, phase, T = stft.encode(mix_aud)

        # Forward — per stereo channel
        mask_l, corr_l = model(mag[:, 0:1], stem_idx)
        mask_r, corr_r = model(mag[:, 1:2], stem_idx)

        full_mask  = torch.cat([mask_l, mask_r], dim=1)
        full_corr  = torch.cat([corr_l, corr_r], dim=1)

        pred_mag   = mag * full_mask
        pred_phase = phase + full_corr
        pred_aud   = stft.decode(pred_mag, pred_phase, length=T)

        # Loss
        loss = loss_fn(pred_aud, tgt_aud)

        # Safety: skip exploding batches
        if torch.isnan(loss) or torch.isinf(loss) or loss.item() > MAX_LOSS:
            skipped += 1
            optimizer.zero_grad()
            del mix_aud, tgt_aud, mag, phase
            del mask_l, mask_r, corr_l, corr_r
            del full_mask, full_corr, pred_mag, pred_phase, pred_aud, loss
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        train_loss_sum += loss.item()

        # Log
        if step % LOG_EVERY == 0:
            sdr = cpu_sdr(pred_aud, tgt_aud)
            train_sdr_sum   += sdr
            train_sdr_count += 1
            lr  = optimizer.param_groups[0]["lr"]
            eta = (time.time() - t0) / (step + 1) * (STEPS_PER_EPOCH - step - 1)
            log.info(
                f"  E{epoch:02d} [{step:03d}/{STEPS_PER_EPOCH}] "
                f"stem={stem:6s} "
                f"loss={loss.item():+7.4f} "
                f"sdr={sdr:+6.2f}dB "
                f"lr={lr:.1e} "
                f"skip={skipped} "
                f"ETA={eta:.0f}s"
            )

        del mix_aud, tgt_aud, mag, phase
        del mask_l, mask_r, corr_l, corr_r
        del full_mask, full_corr, pred_mag, pred_phase, pred_aud, loss

    scheduler.step()

    valid_steps    = STEPS_PER_EPOCH - skipped
    avg_train_loss = train_loss_sum / max(1, valid_steps)
    avg_train_sdr  = train_sdr_sum  / max(1, train_sdr_count)

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    val_sdr_total = 0.0
    val_count     = 0
    stem_sdrs     = {}

    with torch.no_grad():
        for stem in STEM_NAMES:
            s_sdr_sum = 0.0
            s_count   = 0
            s_idx     = torch.tensor([STEM_TO_IDX[stem]], device=device)

            for vi, (mix_aud, tgt_aud) in enumerate(data_mgr.val_loader(stem)):
                if vi >= VAL_BATCHES:
                    break

                mix_aud = mix_aud.to(device)
                tgt_aud = tgt_aud.to(device)

                mag, phase, T = stft.encode(mix_aud)
                mask_l, corr_l = model(mag[:, 0:1], s_idx)
                mask_r, corr_r = model(mag[:, 1:2], s_idx)

                full_mask  = torch.cat([mask_l, mask_r], dim=1)
                full_corr  = torch.cat([corr_l, corr_r], dim=1)
                pred_mag   = mag * full_mask
                pred_phase = phase + full_corr
                pred_aud   = stft.decode(pred_mag, pred_phase, length=T)

                sdr = cpu_sdr(pred_aud, tgt_aud)
                s_sdr_sum     += sdr
                s_count       += 1
                val_sdr_total += sdr
                val_count     += 1

                del mix_aud, tgt_aud, mag, phase
                del mask_l, mask_r, corr_l, corr_r
                del full_mask, full_corr, pred_mag, pred_phase, pred_aud

            stem_sdrs[stem] = s_sdr_sum / max(1, s_count)

    avg_val_sdr = val_sdr_total / max(1, val_count)
    spread      = max(stem_sdrs.values()) - min(stem_sdrs.values())
    elapsed     = time.time() - t0
    lr          = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(avg_train_loss)
    history["val_sdr"].append(avg_val_sdr)
    history["stem_sdrs"].append(stem_sdrs)

    # ── Epoch summary ─────────────────────────────────────────
    log.info("")
    log.info("=" * 65)
    log.info(
        f"EPOCH {epoch:02d}/{MAX_EPOCHS}  "
        f"({elapsed:.0f}s)  lr={lr:.1e}  skipped={skipped}"
    )
    log.info(
        f"  Train : loss={avg_train_loss:+7.4f}  "
        f"sdr={avg_train_sdr:+.2f} dB"
    )
    log.info(f"  Val   : sdr={avg_val_sdr:+.2f} dB")
    log.info(f"  Per-stem:")
    for s, v in stem_sdrs.items():
        tag = " ←best" if v == max(stem_sdrs.values()) else ""
        log.info(f"    {s:8s}: {v:+.2f} dB{tag}")

    steer = "✓ FiLM conditioning active" if spread > 1.0 else "converging"
    log.info(f"  Spread: {spread:.2f} dB  [{steer}]")
    log.info(f"  GPU   : {torch.cuda.memory_allocated()/1e9:.2f} GB")
    log.info("=" * 65)

    # ── Best checkpoint ───────────────────────────────────────
    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        no_improve   = 0
        save_ckpt(epoch, avg_val_sdr, stem_sdrs, history)
        log.info(f"★ NEW BEST → {avg_val_sdr:+.2f} dB  (saved)\n")
    else:
        no_improve += 1
        log.info(f"  No improvement ({no_improve}/{PATIENCE})\n")
        if no_improve >= PATIENCE:
            log.info("Early stopping triggered.")
            break

# ============================================================
# FINAL REPORT
# ============================================================
log.info("")
log.info("=" * 65)
log.info("PHASE 4A TRAINING COMPLETE")
log.info("=" * 65)
log.info(f"Best val SI-SDR : {best_val_sdr:+.2f} dB")
log.info("")

if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu")
    if "stem_sdrs" in ckpt:
        log.info("Best checkpoint per-stem SI-SDR:")
        for s, v in ckpt["stem_sdrs"].items():
            log.info(f"  {s:8s}: {v:+.2f} dB")

log.info("")
log.info("Results comparison:")
log.info(f"  Phase 1  magnitude U-Net    : +3.22 dB  (vocals only)")
log.info(f"  Phase 2  + Harmonic GCN     : +3.12 dB  (vocals only, -31% params)")
log.info(f"  Phase 3  + CLAP single stem : +3.80 dB  (vocals only)")
log.info(f"  Phase 4A multi-stem complex : {best_val_sdr:+.2f} dB  (all 4 stems)")
log.info("")

if best_val_sdr > 0:
    log.info("✓ Multi-stem separation working!")
    log.info("  → Next: Phase 4B adds CLAP text steering on top")
else:
    log.info("Still learning. Run again to auto-resume from best checkpoint.")

23:14:07 | GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU
23:14:07 | VRAM: 6.4 GB
23:14:07 | Model parameters: 2,434,275
23:14:07 | Loaded 106 Phase 2 tensors
23:14:08 | Sanity check...



MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)


23:14:09 |   Mask range  : [0.174, 0.810]
23:14:09 |   Phase corr  : [-0.1000, -0.0211]
23:14:09 |   Loss        : 2.4295
23:14:09 |   SDR         : -5.78 dB
23:14:09 |   ✓ Passed

23:14:09 | =================================================================
23:14:09 | GSN PHASE 4A — MULTI-STEM SEPARATION (NO CLAP)
23:14:09 |   Epochs    : 1 → 30
23:14:09 |   Steps     : 400 per epoch (1600 updates)
23:14:09 |   Stems     : ['vocals', 'drums', 'bass', 'other']
23:14:09 |   Condition : FiLM (learned stem embedding)
23:14:09 |   Loss      : log-STFT + SI-SNR[-10,30] + waveform-L1
23:14:09 |   LR        : 5e-05 (CosineAnnealing)
23:14:09 |   Grad clip : 0.5
23:14:09 |   Patience  : 10
23:14:09 | =================================================================

23:14:10 |   E01 [000/400] stem=vocals loss=+4.0616 sdr=-60.00dB lr=5.0e-05 skip=0 ETA=447s



MUSDB18Dataset  split=train  target=drums
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=train  target=bass
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=train  target=other
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)


23:14:29 |   E01 [050/400] stem=bass   loss=-1.7675 sdr= +4.72dB lr=5.0e-05 skip=0 ETA=135s
23:14:49 |   E01 [100/400] stem=vocals loss=+4.1150 sdr=-14.23dB lr=5.0e-05 skip=0 ETA=116s
23:15:08 |   E01 [150/400] stem=bass   loss=+1.4576 sdr= -3.36dB lr=5.0e-05 skip=0 ETA=96s
23:15:26 |   E01 [200/400] stem=vocals loss=+4.1483 sdr=-60.00dB lr=5.0e-05 skip=0 ETA=76s
23:15:45 |   E01 [250/400] stem=bass   loss=+3.6208 sdr= -8.83dB lr=5.0e-05 skip=0 ETA=57s
23:16:03 |   E01 [300/400] stem=vocals loss=+4.1250 sdr=-13.45dB lr=5.0e-05 skip=0 ETA=37s
23:16:20 |   E01 [350/400] stem=bass   loss=+1.7460 sdr= -4.06dB lr=5.0e-05 skip=0 ETA=18s



MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=drums
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=bass
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=other
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)


23:16:42 | 
23:16:42 | =================================================================
23:16:42 | EPOCH 01/30  (153s)  lr=5.0e-05  skipped=0
23:16:42 |   Train : loss=+2.4285  sdr=-19.90 dB
23:16:42 |   Val   : sdr=-8.28 dB
23:16:42 |   Per-stem:
23:16:42 |     vocals  : -8.52 dB
23:16:42 |     drums   : -9.48 dB
23:16:42 |     bass    : -10.97 dB
23:16:42 |     other   : -4.16 dB ←best
23:16:42 |   Spread: 6.81 dB  [✓ FiLM conditioning active]
23:16:42 |   GPU   : 0.07 GB
23:16:42 | =================================================================
23:16:42 | ★ NEW BEST → -8.28 dB  (saved)

23:16:42 |   E02 [000/400] stem=vocals loss=+1.3853 sdr= -3.20dB lr=5.0e-05 skip=0 ETA=141s
23:17:15 |   E02 [050/400] stem=bass   loss=+1.6633 sdr= -3.82dB lr=5.0e-05 skip=0 ETA=226s
23:17:48 |   E02 [100/400] stem=vocals loss=-1.0118 sdr= +2.78dB lr=5.0e-05 skip=0 ETA=196s
23:18:21 |   E02 [150/400] stem=bass   loss=+2.2239 sdr= -5.40dB lr=5.0e-05 skip=0 ETA=163s
23:18:54 |   E02 [200/400] stem=

KeyboardInterrupt: 